
# Diabetes Prediction Using NHANES Dataset

This notebook demonstrates how to preprocess the NHANES 2017–2018 dataset and apply machine learning models to predict diabetes. We will use Logistic Regression, Random Forest, and XGBoost classifiers and evaluate their performance.


In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

# Simulate NHANES-like dataset
np.random.seed(42)
n = 1000
data = pd.DataFrame({
    'RIDAGEYR': np.random.randint(18, 80, n),
    'RIDETH3': np.random.choice([1, 2, 3, 4, 5], n),
    'DMDEUC2': np.random.choice([1, 2, 3, 4], n),
    'INDFMPIR': np.random.uniform(0.5, 5.0, n),
    'DIQ010': np.random.choice([0, 1], n),
    'DIQ050': np.random.choice([0, 1], n),
    'DIQ070': np.random.choice([0, 1], n),
    'LBXTC': np.random.uniform(100, 300, n),
    'BMXBMI': np.random.uniform(18, 40, n),
    'BPXSY1': np.random.uniform(90, 180, n)
})

# Display first few rows
data.head()


In [ ]:

# Target and features
y = data['DIQ010']
X = data.drop(columns=['DIQ010'])

# One-hot encoding
X_encoded = pd.get_dummies(X, columns=['RIDETH3', 'DMDEUC2'], drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:

# Train models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

results = {}
roc_data = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = classification_report(y_test, y_pred, output_dict=True)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = {'fpr': fpr, 'tpr': tpr}

# Display classification reports
for name in results:
    print(f"
{name} Classification Report:")
    print(pd.DataFrame(results[name]).transpose())


In [ ]:

# Feature importance from Random Forest
rf_importances = models['Random Forest'].feature_importances_
features = X_train.columns
importance_df = pd.DataFrame({'Feature': features, 'Importance': rf_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance - Random Forest')
plt.tight_layout()
plt.show()


In [ ]:

# Plot ROC curves
plt.figure(figsize=(8,6))
for name, roc in roc_data.items():
    plt.plot(roc['fpr'], roc['tpr'], label=name)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()
